# 01 - Data Ingestion
Load the source data and ingest it into persistent bronze layer.

**Raw data -> Ingestion -> Bronze tables** (Can be used in other notebooks without loading the csv files)

## Import pyspark

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
RAW_PATH = "/Workspace/Users/yogeshberwal266@gmail.com/Capstone Project/data/raw"
print(RAW_PATH)

In [0]:
pwd

## Reading source files

In [0]:
stores_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/stores.csv")
)

In [0]:
stores_df.show()

In [0]:
stores_df.printSchema()


In [0]:
print("Rows:", stores_df.count())
print("Columns:", len(stores_df.columns))

## Loading all 6 database
Raw csv -> dataframe

In [0]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/products.csv")
)

suppliers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/suppliers.csv")
)

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/sales_transactions.csv")
)

inventory_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/inventory.csv")
)

purchase_orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/purchase_orders.csv")
)

In [0]:
datasets = {
    "stores": stores_df,
    "products": products_df,
    "suppliers": suppliers_df,
    "sales": sales_df,
    "inventory": inventory_df,
    "purchase_orders": purchase_orders_df
}

for name, df in datasets.items():
    print(f"{name:20} rows={df.count():5} columns={len(df.columns)}")
    


In [0]:
for name, df in datasets.items():
    print("=" * 40)
    print(name.upper())
    df.printSchema()
    df.show()

## Saving bronze data into tables
data frame -> persistent Bronze delta tables

In [0]:
stores_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("stores_bronze")

products_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("products_bronze")

suppliers_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("suppliers_bronze")

sales_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("sales_bronze")

inventory_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("inventory_bronze")

purchase_orders_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("purchase_orders_bronze")

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SELECT * FROM stores_bronze

In [0]:
%sql
SELECT *
FROM products_bronze
WHERE unit_price <= 0;

In [0]:
%sql
SELECT product_id, COUNT(*) as duplicate_count
FROM products_bronze
GROUP BY product_id
HAVING COUNT(*) > 1
